# Assignment 10 - Problem 2
## Running DeepSeek-R1 locally with Ollama

**Name:** Hussein Jawad

The task: install Ollama, download (pull) the DeepSeek-R1 model, run it and check with `ollama ps` that it is running, then use `ollama.chat()` from Python to show that the model gives an answer together with its chain of thought.

The question I ask the model:

> "Explain the difference between the General and Special theory of relativity."

Contents:
- Step 1 - Install Ollama
- Step 2 - Check my machine and choose the model size
- Step 3 - Pull the DeepSeek-R1 model
- Step 4 - Run the model and check it with `ollama ps`
- Step 5 - Ask the question with `ollama.chat()` and show the chain of thought
- Observations
- Appendix - Concepts used in this problem

## Step 1 - Install Ollama

Ollama runs large language models on my own computer. I downloaded the installer from https://ollama.com/download and ran it. After the install, Ollama runs in the background and listens on `http://localhost:11434`. Both the `ollama` command and the Python library talk to it there.

I also install the Ollama Python library, which gives me `ollama.chat()`.

In [1]:
# Install the Ollama Python library (for ollama.chat) and psutil (to read the machine's memory)
%pip install -q ollama psutil

Note: you may need to restart the kernel to use updated packages.


Check that Ollama is installed (the `!` runs a terminal command from the notebook):

In [2]:
# Show the installed Ollama version
!ollama --version

ollama version is 0.40.16


## Step 2 - Check my machine and choose the model size

DeepSeek-R1 comes in several sizes (1.5b, 7b, 8b, 14b, 32b, 70b, 671b; b = billions of parameters). From the lab, the rule of thumb is that I need at least as much free memory (RAM, or GPU memory for speed) as the size of the download, plus some extra for the context. `deepseek-r1:8b` is a 5.2 GB download, so a computer with about 8 GB of RAM or more can run it.

In [3]:
import platform, shutil, subprocess   # operating system info, find programs, run commands
import psutil                         # memory info

print("Operating system:", platform.system(), platform.release())                 # e.g. Windows 11
print("CPU cores       :", psutil.cpu_count(logical=True))                         # number of CPU threads
print("RAM total       :", round(psutil.virtual_memory().total / 1e9, 1), "GB")    # installed memory
print("RAM available   :", round(psutil.virtual_memory().available / 1e9, 1), "GB")  # memory free right now

if shutil.which("nvidia-smi"):                                                       # an NVIDIA GPU driver is installed
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True).stdout.strip()             # ask the driver for name and memory
    print("GPU             :", gpu)                                                 # e.g. NVIDIA L4, 23034 MiB
else:
    print("GPU             : no NVIDIA GPU found (the model will run on the CPU, which is slower)")

Operating system: Windows 11
CPU cores       : 12
RAM total       : 16.8 GB
RAM available   : 2.0 GB
GPU             : no NVIDIA GPU found (the model will run on the CPU, which is slower)


My PC has 16.8 GB of RAM and no NVIDIA GPU, so the model runs on the CPU. 16.8 GB is enough for the 8b model, so I used `deepseek-r1:8b`, the size the assignment recommends. The bigger models (14b is about a 9 GB download, 32b about 20 GB) would be even slower on a CPU or wouldn't fit. Only 2.0 GB of RAM was free at that moment because other programs were open, but the model still loaded in Step 4.

In [4]:
MODEL = "deepseek-r1:8b"   # 8B is the size the assignment recommends; use "deepseek-r1:1.5b" if memory is low
print("Model:", MODEL)     # the model used in the rest of the notebook

Model: deepseek-r1:8b


## Step 3 - Pull the DeepSeek-R1 model

`ollama.pull()` downloads the model, the same as `ollama pull deepseek-r1:8b` in a terminal. The first time it downloads about 5.2 GB. If the model is already on the computer, it only checks it. I print each stage once to keep the output short.

In [5]:
import ollama   # Ollama Python library: talks to the Ollama server on localhost:11434

last_status = None                                  # last stage printed
for progress in ollama.pull(MODEL, stream=True):    # download the model, receiving progress updates
    if progress.status != last_status:              # print each new stage only once
        print(progress.status)
        last_status = progress.status

pulling manifest
pulling e6a7edc1a4d7
pulling c5ad996bda6e
pulling 6e4c38e1172f
pulling ed8474dc73db
pulling f64cd5418e4b
verifying sha256 digest
writing manifest
removing unused layers
success


## Step 4 - Run the model and check it with `ollama ps`

Sending an empty prompt loads the model into memory without asking it anything. `keep_alive="30m"` keeps it loaded for 30 minutes (the default is 5), so it is still running for the question in Step 5. Then `ollama ps` shows which models are running.

In [7]:
import time                                                     # to measure how long things take

start = time.time()                                             # start the timer
ollama.generate(model=MODEL, prompt="", keep_alive="30m")       # empty prompt = just load the model into memory
print(f"Model loaded in {time.time() - start:.1f} seconds")     # loading time

# Show the models that are running right now
!ollama ps

Model loaded in 17.2 seconds
NAME              ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL               
deepseek-r1:8b    6995872bfe4c    5.9 GB    100% CPU     4096       llamacpp    29 minutes from now    


`ollama ps` shows that `deepseek-r1:8b` is running:
- **SIZE 5.9 GB:** memory in use, more than the 5.2 GB download because it includes memory for the context
- **PROCESSOR 100% CPU:** the whole model runs on the CPU, because my PC has no NVIDIA GPU
- **CONTEXT 4096:** the model can look at 4,096 tokens
- **UNTIL 29 minutes from now:** it stays loaded because of `keep_alive="30m"`

## Step 5 - Ask the question with `ollama.chat()` and show the chain of thought

`ollama.chat()` takes the model name and a list of messages. With `think=True`, DeepSeek-R1 returns its reasoning in a separate field:
- `response.message.thinking` - the chain of thought
- `response.message.content` - the final answer

The model first writes all of its thinking and then the answer, so this call takes a while. On my PC, which has no GPU, it took about 16 minutes.

In [9]:
QUESTION = "Explain the difference between the General and Special theory of relativity."   # my question

start = time.time()                                          # start the timer
response = ollama.chat(
    model=MODEL,                                             # deepseek-r1:8b
    messages=[{"role": "user", "content": QUESTION}],       # one user message
    think=True,                                              # return the chain of thought separately
    keep_alive="30m",                                        # keep the model loaded afterwards
)
seconds = time.time() - start                                # how long the whole call took

thinking = response.message.thinking or ""                   # chain of thought (empty if the model gave none)
answer = response.message.content                            # final answer

print(f"Time                : {seconds:.1f} s")                                        # wall-clock time
print(f"Prompt tokens       : {response.prompt_eval_count}")                           # tokens in my question
print(f"Generated tokens    : {response.eval_count}")                                  # thinking + answer tokens
print(f"Speed               : {response.eval_count / (response.eval_duration / 1e9):.1f} tokens/s")   # durations are in ns
print(f"Chain of thought    : {len(thinking):,} characters")                           # length of the thinking
print(f"Final answer        : {len(answer):,} characters")                             # length of the answer
print(f"Thinking kept apart : {'<think>' not in answer and len(thinking) > 0}")       # True = two separate fields

Time                : 981.5 s
Prompt tokens       : 17
Generated tokens    : 2193
Speed               : 2.3 tokens/s
Chain of thought    : 4,174 characters
Final answer        : 6,366 characters
Thinking kept apart : True


### The chain of thought

The thinking is long, so I print the beginning and the end of it.

In [10]:
if len(thinking) > 2500:                                      # long: show the start and the end only
    print(thinking[:1800])                                    # first 1,800 characters
    print(f"\n[... {len(thinking) - 2500:,} characters skipped ...]\n")
    print(thinking[-700:])                                    # last 700 characters
else:
    print(thinking)                                           # short enough to show in full


Okay, the user asked to explain the difference between General and Special Relativity. Let me start by recalling what I know. Special Relativity is about constant speed, no gravity, while General adds gravity and acceleration.

Hmm, the user might be a student studying physics, or maybe someone curious about these concepts. They probably have a basic understanding but need a clear distinction. I should break down the key points without too much jargon.

First, I need to mention the two postulates of Special Relativity. The invariance of the speed of light and the principle of relativity. That's foundational. Then, the main effects like time dilation and length contraction. Maybe include examples like muons or light travel to make it relatable.

Wait, the user might not know what postulates are, so I should explain them simply. Also, they might not grasp why the speed of light is a universal constant. Emphasize that it's the same for all observers, which is counterintuitive.

Now, Gene

### The final answer

In [11]:
print(answer)   # the model's final answer, without the thinking


Okay, let's break down the difference between Special Relativity (SR) and General Relativity (GR). Einstein developed SR in 1905 and GR in 1915, with GR building upon the foundations of SR.

Here's the comparison:

## Special Relativity (1905)

*   **Scope:** Deals with physics in **inertial reference frames** (frames moving at constant velocity, not accelerating). It does not include gravity.
*   **Core Idea:** The laws of physics are the same for all observers in uniform motion (constant velocity), and the speed of light in a vacuum is constant for all observers, regardless of the motion of the light source or observer.
*   **Key Postulates:**
    1.  The principle of relativity: The laws of physics are identical in all inertial reference frames.
    2.  The invariance of the speed of light: The speed of light in a vacuum (c) is the same for all observers, even those moving relative to each other.
*   **Main Effects:** Focuses on how space and time are perceived differently by obser

## Observations

**Ollama and the model.** Ollama 0.40.16 is installed and running on my Windows 11 PC. `ollama.pull()` finished with `success`, so `deepseek-r1:8b` (5.2 GB) is on my PC.

**The model is running.** After it loaded (17.2 s), `ollama ps` listed `deepseek-r1:8b` with 5.9 GB in memory, `100% CPU`, a 4,096-token context and 29 minutes left before it unloads.

**Speed.** The question took 981.5 s (about 16 minutes). The model generated 2,193 tokens at 2.3 tokens per second, so almost all of the time went into writing them (2,193 / 2.3 ≈ 950 s). It is slow because the model runs only on the CPU and, as a reasoning model, it writes its thinking before the answer. A GPU, or the smaller `deepseek-r1:1.5b`, would be much faster. Closing other programs first would also leave more memory free (only 2.0 GB was free in Step 2).

**Chain of thought.** `think=True` returned the reasoning separately: 4,174 characters in `message.thinking` and 6,366 characters in `message.content`, with no `<think>` tags in the answer (the check printed `True`). The thinking reads like a plan. The model recalls the main idea (special relativity = constant speed and no gravity; general relativity adds gravity and acceleration), guesses who is asking (probably a student), and plans the structure: each theory on its own, then a comparison. The final answer follows that plan, with a section for each theory and a summary table. The thinking also mentions examples it wanted to use (muons, an elevator, a trampoline), but none of them appear in the final answer. So the chain of thought is a plan, not an exact script of the answer.

**Is the answer correct?** Mostly yes. Compared with the table in the appendix, it has nearly all the main points (it doesn't mention black holes): the two postulates of special relativity, time dilation, length contraction, E = mc² and flat spacetime. For general relativity it covers the equivalence principle, gravity as curved spacetime, light bending (1919 eclipse), gravitational time dilation (GPS), Mercury's orbit and gravitational waves. Three things are simplified:
- It says special relativity "doesn't handle" accelerating frames. That is a common simplification: special relativity can describe accelerating objects. What it can't describe is gravity, so the real difference is flat spacetime (special) vs curved spacetime (general).
- It writes the field equations as Gᵘᵛ = 8πTᵘᵛ, which is only true in units where G = c = 1. Normally they are G<sub>μν</sub> = (8πG / c⁴) T<sub>μν</sub>.
- Its airplane-clock example only counts the speed effect. At cruising altitude gravity is weaker, which makes the clock run faster, so in real measurements (the Hafele-Keating experiment, 1971) the result depended on the direction of the flight.

**Conclusion.** DeepSeek-R1 runs locally with Ollama, `ollama ps` confirms that it is loaded, and `ollama.chat()` returns both the chain of thought and the final answer. On a PC without a GPU it works, but it is slow: about 16 minutes for one answer.

## Appendix - Concepts used in this problem

### 1. Running a model locally

**Ollama.** A program that downloads and runs open LLMs on my own computer. It runs a small server on `http://localhost:11434`; the `ollama` command, `curl` and the Python library all send their requests there. Nothing leaves my computer, and there is no API key and no cost per request.

**Model name and tag.** `deepseek-r1:8b` = model `deepseek-r1`, tag `8b` (8 billion parameters). A name without a tag means `:latest`.

*Here:* `deepseek-r1:8b`, 5.2 GB on disk.

**Distilled models.** The full DeepSeek-R1 has 671B parameters, far too big for a laptop. The smaller versions (1.5b to 70b) are *distilled*: smaller open models trained to copy the reasoning of the big one.

**Quantization.** Storing each weight with fewer bits (Ollama models use about 4 bits by default). That is why an 8B model is only about a 5 GB download instead of about 16 GB at 16 bits per weight.

**Memory, GPU and CPU.** The whole model has to fit in memory. If it fits in GPU memory it runs fast; if not, part of it runs on the CPU, which is much slower.

*Here:* my PC has no NVIDIA GPU, so `ollama ps` showed `100% CPU` and the model ran at 2.3 tokens/s.

### 2. The commands I used

| Command | What it does |
|---|---|
| `ollama --version` | shows that Ollama is installed |
| `ollama pull <model>` / `ollama.pull()` | downloads a model |
| `ollama ps` | lists the models loaded in memory right now |
| `ollama.generate(model, prompt="")` | loads a model without asking anything |
| `ollama.chat(model, messages, think=True)` | asks a question and returns thinking + answer |

How to read `ollama ps`:

| Column | Meaning |
|---|---|
| NAME | the model that is loaded |
| SIZE | memory used: the weights plus memory for the context, so more than the download size |
| PROCESSOR | `100% GPU` = all on the GPU (fast); a CPU share means part of it runs on the CPU (slower) |
| CONTEXT | how many tokens the model can look at |
| RUNNER | the engine that runs the model (here `llamacpp`, i.e. llama.cpp) |
| UNTIL | when Ollama will unload the model (`keep_alive`, 5 minutes by default) |

### 3. Reasoning and the chain of thought

**Reasoning model.** DeepSeek-R1 is trained to "think out loud" before it answers: it writes a chain of thought (working out the problem step by step, checking itself), and then the final answer. This often makes answers better on harder questions, but it costs many extra tokens, so it is slower.

**`think=True`.** Tells Ollama to return the thinking in `message.thinking` and only the answer in `message.content`. Older versions put the thinking inside `<think>...</think>` tags in the content.

**Response numbers.** `prompt_eval_count` = tokens in my question, `eval_count` = tokens the model generated (thinking + answer), `eval_duration` and `total_duration` are in nanoseconds. Speed = `eval_count / (eval_duration / 10^9)` tokens per second.

*Here:* 17 prompt tokens, 2,193 generated tokens, 2.3 tokens/s, 981.5 s in total.

### 4. The question: Special vs General relativity

To judge whether the model's answer is right, the main points are:

| | Special relativity (1905) | General relativity (1915) |
|---|---|---|
| Covers | flat spacetime: no gravity. Usually described from frames moving at constant velocity (inertial frames), but it can also describe accelerating objects | gravity, described as curved spacetime; works in any frame, including accelerating ones |
| Key ideas | the laws of physics are the same in all inertial frames; the speed of light is the same for every observer | gravity is the curvature of spacetime caused by mass and energy; equivalence principle (gravity and acceleration feel the same) |
| Results | time dilation, length contraction, E = mc² | light bending around the Sun, gravitational time dilation (GPS corrections), black holes, gravitational waves |

The main difference is flat vs curved spacetime. Special relativity is the special case of general relativity when gravity can be ignored.